# 🎯 WavLM Extraction — 221 EMNLP Videos
**Audio:** `/content/drive/MyDrive/standup4ai/audio_1000/{vid}.m4a`
**Output:** `/content/drive/MyDrive/standup4ai/features_221/`
**Checkpoint:** Every 10 videos to Drive

**Features:** WavLM 768-dim + prosody 23-dim = 791-dim per 5s chunk
**GPU:** Colab T4


In [ ]:
# Cell 1: Setup + Mount Drive + Install deps
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, subprocess

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = f'{BASE}/features_221'
AUDIO_DIR = f'{BASE}/audio_1000'

os.makedirs(FEAT_DIR, exist_ok=True)

# Verify audio dir
if os.path.exists(AUDIO_DIR):
    n_audio = len([f for f in os.listdir(AUDIO_DIR) if f.endswith('.m4a')])
    print(f'Audio dir: {AUDIO_DIR} ({n_audio} files)')
else:
    print(f'WARNING: Audio dir not found: {AUDIO_DIR}')

# Install deps
subprocess.run(['pip', 'install', 'soundfile', '-q'], capture_output=True)
print('Setup complete')

In [ ]:
# Cell 2: Load WavLM on GPU
import torch
from transformers import AutoModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('WavLM ready on GPU')

In [ ]:
# Cell 3: Prosody extractor (23-dim)
import numpy as np
import librosa

def prosody23(y, sr):
    f = np.zeros(23, dtype=np.float32)
    # F0 (5 dims)
    try:
        f0, v, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        fc = f0[~np.isnan(f0)].astype(np.float32)
        vc = v[~np.isnan(f0)].astype(np.float32)
        if len(fc) > 0:
            f[0] = np.mean(fc); f[1] = np.std(fc)
            f[2] = np.max(fc); f[3] = np.min(fc)
            f[4] = np.mean(vc)
    except:
        pass
    # Energy (5 dims)
    hop = 512
    try:
        rms = librosa.feature.rms(y=y, hop_length=hop)[0].astype(np.float32)
        f[5] = np.mean(rms); f[6] = np.std(rms)
        f[7] = np.max(rms); f[8] = np.min(rms)
        f[9] = f[7] - f[8]
    except:
        pass
    # Duration (2 dims)
    f[10] = len(y) / sr
    try:
        rms_m = np.mean(rms)
        f[11] = f[10] / (np.sum(rms > rms_m) + 1)
    except:
        pass
    # Spectral (5 dims)
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0].astype(np.float32)
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0].astype(np.float32)
        sf = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0].astype(np.float32)
        z = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0].astype(np.float32)
        f[12] = np.mean(sc); f[13] = np.mean(sb)
        f[14] = np.mean(sf); f[15] = np.mean(z); f[16] = np.std(z)
    except:
        pass
    # Voice quality (6 dims)
    try:
        yh, _ = librosa.effects.hpss(y.astype(np.float32))
        f[17] = np.mean(np.abs(yh)) / (np.mean(np.abs(y)) + 1e-8)
        f[18] = np.mean(np.abs(y)); f[19] = np.std(y); f[20] = np.max(np.abs(y))
    except:
        pass
    return f

def extract_features(audio_path, vid):
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        cs = 16000 * 5  # 5-second chunks
        n = len(y) // cs
        if n == 0:
            print(f'  {vid}: Audio too short ({len(y)/sr:.1f}s)')
            return None
        feats = []
        for i in range(n):
            ch = y[i*cs:(i+1)*cs].astype(np.float32)
            # WavLM
            t = torch.tensor(ch).unsqueeze(0).to(device)
            with torch.no_grad():
                r = wavlm(t).last_hidden_state.mean(dim=2).squeeze().cpu().numpy().astype(np.float32)
            # Prosody
            p = prosody23(ch, 16000)
            feats.append(np.concatenate([r, p]))  # 768 + 23 = 791
        return np.array(feats, dtype=np.float32)
    except Exception as e:
        print(f'  {vid}: Error - {e}')
        return None

print('Extractors ready')

In [ ]:
# Cell 4: Process all 221 videos
import time, json

VIDEO_IDS = ["18H1aeoGybw", "18rLwnvxOU0", "21gOjz-Xk7s", "41piF6uPhXg", "482LeT9UT7I", "53JXuJGmhoU", "5bKcTy3zag4", "5cdoHY0ziVA", "5gp79fSWHy0", "66CyaeFWucM", "6Ofc2A75zuw", "76r8IcowEsE", "7E7la6BCpRc", "7Gw1NjZ13fA", "7VkAFkK3bwQ", "7cBFWZDXlHA", "7gRo0nF1yS0", "7kULz2NevT4", "8CoHAczz9pY", "8EUpV_qyEpc", "8eYSNXOsyoo", "8nltoWdciws", "90s9HfZhM0Y", "9DwiBEVDdUE", "9h7-OMYItDI", "9yPco6WNYG0", "AES4jzE513Y", "AEnlxaPVtK8", "AI69HZWZ26c", "A_EIL1ojfK4", "Azl5GJuYqE0", "B9jLEExvazc", "BT-WOZQ5JRc", "BbBymwvs7co", "Bl-PyS4f8as", "BoMFeYyvYP8", "C1TMo0YTDLA", "CNKnRGig1FM", "CUEvqRwSi_c", "CgeJOMi1mEU", "CocEMvDXiu8", "CwMCoAh1-a0", "CwYov1S2060", "DN-dSUoKuJQ", "DN-vrVTIKwk", "DcwbxzMMPEM", "DlS85HEFvlE", "DsSh_4VSXP4", "E8ToU_gqdlY", "EIpbdW9Vb6s", "EPY7mNtnKy8", "EtdeJ-2bYO0", "Eva3iridbd4", "FBX2tyYrjog", "FDKzqoSjFxs", "FUbXP88a0wk", "FcOGp4y2O3I", "GCfX2_fRSIk", "G_hDEQDtEf0", "Ga1dQxpUft0", "H3Y-9-CarcQ", "HIooOHs_8sg", "HKJKaIwCCQk", "HjTy45N6G6E", "HtPGDsZEIC0", "IFtWoIw0DVE", "IIhav6q5IsE", "INhj_TbMRXk", "JIWQBC8Q1e8", "JLOjHhWTKLA", "JMbjs8fLyaY", "JaRBJnElNZc", "JsL-FKqlHD0", "Jw_jJ8fE9ZA", "KRMi6FzC2vM", "KfSI5Krer8g", "KicH8MlWxn4", "KqpzX1oH-E0", "Ku-mGC3pRsk", "LH6JJn5PPWg", "LYXsdcLkIVY", "M1NDZYLSo94", "M5Utz5IROls", "MA6B8NQ9oxo", "MBmldf0UW-A", "MHLcCWGpyRc", "NurQtpa590E", "NwKrc00JPVA", "OESjhjRYBUQ", "OZnoJ9WoLGw", "O_YGnqj_z2M", "Obc1d4v4x2U", "P0-fCn2ptvc", "PQyFD9DRKpk", "PVaNYuVeRE0", "Q1EytxIpFIo", "QJ6tmMFp0Rs", "QO8QC0QpPr4", "QRC1wN7dwPM", "R5HTUMhnabA", "RuciD4LJo6Q", "S18ykYyYEr4", "S2SXqVjmJ9E", "S5oYYq2KPoc", "Sfv6oiA89O8", "Sl8Zazk2C3s", "SofFDLTpy68", "TdAMvTod6zQ", "UI2Bb1zAFw4", "UPgE_HxS88Q", "Ucz9sCT6ydQ", "Utog2sKMYOI", "VLyPoVtMxzg", "VSytmEkhOdA", "Vle0vKahRzA", "VxNJxDKkLGo", "W3jHBME1C0M", "WN9u1jSMRGM", "WrklYFQIYRI", "WtfN5loZa08", "XKGDf_btalc", "XMwZnO7teGA", "X_MsBNNePpw", "YAqJM9QFmgo", "YZzO06XQAw0", "Z1ChLkF6ooE", "ZV8Xgf0i1gE", "ZpF_bEUkqnk", "ZxsuaSEKDyk", "a7WbsSPVQwQ", "aU7VUkOLx6s", "apSIp7FvEwI", "aqnqgMxfpFo", "ayMuQVc8DWA", "bOYGQFO65c0", "bUf7sYCYI6A", "bXKT1OedpZk", "cObKxofJ1ww", "cl2qN3mbfXs", "e5p8aXTgFgo", "em6rlnYLspg", "f-eadBMRiMw", "fQjbi5VRNs4", "gOniIHA6xqU", "gVw7B-EXNfo", "gXjE2gqcs0U", "hRRvWVTHK8k", "hqAuogvh8Rk", "i8VjBdvQ0Ko", "i9UsxfvOzSs", "iym4rS5WT8U", "izYxUn2WXfc", "jF6Devdvzqo", "joBWJ2b457o", "k1Tl-uNpey4", "kMiPVkw8dlQ", "kMkoV4xoadQ", "kjH9M7Mkwzk", "kwV4zErO5jw", "l2oaxKORheA", "lG-W3DNL4Ps", "lGjXQbZ_GAM", "lNi9kBtraPE", "lZJinxkxvOc", "l_UVH2OJOIc", "lgySVdjX86E", "lnd9QY-Sa20", "lzKSw7679PU", "m8X0F6diPDg", "mA_yzRl5bfc", "mG0SEr6jZSo", "ml-wZ-Os-04", "n3aBK6vd4rM", "nJK5wIkM7V0", "nQpaY1-_LMY", "nlol_hJwEr8", "oGWVsPp4a5E", "pV_KOZ1jJTU", "pdyGbfrMyLo", "pnBa8uvzqtQ", "q112mLKiUCw", "qCilT_NrKac", "qyFw-BnDyes", "rAPZ26R8on8", "rHfWihSHAGQ", "rK_FOkWcwG4", "rQKN_sp00O8", "rRYPyKo_1l8", "rRYa8Cd8NSE", "rjiH4VcYrnA", "sVxPD081Xa8", "spZC_hrHTe0", "tG2qlpdNPbo", "tJKiK1WcA8s", "t_1ULTlaZ4I", "u7eWgKgRau8", "uORo9BRI1EQ", "v7USDAkEzdE", "vc37gyX0F7g", "viSnHyBxYAA", "vn9i61wzYLQ", "vtVfYS3c1RU", "vuJfIw425f8", "w3AzeGKnNAY", "wM1BECRNygc", "wReG9acljeU", "wUSiztgPM8A", "wd4K5OJU_BU", "wwCxu2yJfSU", "x1iBps71YPs", "x9T-oDFcb1I", "xXi_AAgaZOo", "xh06K8YYyN8", "xlbl--hurDQ", "yYAjshQA2ms", "yYdkqNt4azU", "z2NcUVLwH-Y", "z4Wht3FpzPg", "zEcWVrCAk0A", "zNOZzpm3fz8", "zfMg7k1swX8"]
print(f'Total videos: {{len(VIDEO_IDS)}}')

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = f'{{BASE}}/features_221'
AUDIO_DIR = f'{{BASE}}/audio_1000'

# Load checkpoint
ckpt_file = f'{{BASE}}/features_221_checkpoint.json'
done = []
if os.path.exists(ckpt_file):
    with open(ckpt_file) as f:
        done = json.load(f).get('done', [])
done_set = set(done)
print(f'Already done: {{len(done_set)}}')

t0 = time.time()
failed = 0

for i, vid in enumerate(VIDEO_IDS):
    if vid in done_set:
        continue
    
    # Audio path
    audio_path = f'{{AUDIO_DIR}}/{{vid}}.m4a'
    
    if not os.path.exists(audio_path):
        print(f'{{i+1}}/{{len(VIDEO_IDS)}} {{vid}}: AUDIO NOT FOUND')
        failed += 1
        continue
    
    # Extract
    feats = extract_features(audio_path, vid)
    
    if feats is not None:
        np.save(f'{{FEAT_DIR}}/{{vid}}_features.npy', feats)
        done.append(vid)
        done_set.add(vid)
        
        # Save checkpoint every 10 videos
        if len(done_set) % 10 == 0:
            with open(ckpt_file, 'w') as f:
                json.dump({{'done': done}}, f)
        
        elapsed = time.time() - t0
        rate = len(done_set) / elapsed * 3600 if elapsed > 0 else 0
        print(f'{{i+1}}/{{len(VIDEO_IDS)}} {{vid}}: {{feats.shape}} ({{len(done_set)}} done, {{rate:.0f}}/hr, {{failed}} failed)')
    else:
        failed += 1

# Final checkpoint
with open(ckpt_file, 'w') as f:
    json.dump({{'done': done}}, f)

print(f'\nDone: {{len(done_set)}}/{{len(VIDEO_IDS)}} in {{(time.time()-t0)/60:.0f}} min ({{failed}} failed)')

In [ ]:
# Cell 5: Summary
BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = f'{BASE}/features_221'

feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])
print(f'Total features: {len(feat_files)} videos')

import numpy as np
for f in feat_files[:5]:
    d = np.load(f'{FEAT_DIR}/{f}')
    print(f'  {f}: {d.shape}')

print('\nDone!')